<a href="https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

Two parts: (1) I read two findings from FlyRank's own research paper the way I'd want my own work
read, and write the methodology question each one raises. (2) I turn that same lens on my own
Week-5 model — an honest-split before/after, a leakage audit, and a rewrite of my boldest claim.

## 1. Two paper findings + my methodology questions

Source: `docs/flyrank-seo-research-march-2026.pdf`, "The State of AI-Driven SEO" (March 2026),
341,701 content pieces / 57 brands, ML appendix on a 61.8K-row sampled feature-vector subset.

**Finding A — "What Predicts Health?" (ML Appendix, p.27).** A Random Forest ranks feature
importance for predicting Health Score: Average Position 43%, Impressions 32%, Scroll Depth 15%,
CTR 8%. The paper itself is upfront that this is a caveat, not a clean result: *"the target itself
is partly constructed from some of these inputs, so importance is descriptive rather than
causal."* Health Score's own stated formula (Methodology, p.36) is `Impressions (30) + Position
(30) + CTR (20) + Scroll Depth (20)` — so two of the top three "predictors" are literally addends
of the label.

**My methodology question:** the paper names this as a caveat but doesn't run the confirming test —
per the leakage taxonomy's own diagnostic ("train once WITH the suspect, once WITHOUT — a collapse
from ~1.0 toward baseline is the confession"), what does feature importance / model score look
like with Average Position and Impressions *removed*, leaving only the inputs that aren't part of
the label formula (Scroll Depth, CTR component aside)? Without that ablation, a reader can't tell
how much of the 43%+32% is "the model found something real about position and traffic" versus
"the model rediscovered its own label's addition formula." This isn't a flaw unique to this paper —
it's exactly the check I run on my own model in Section 3 below.

**Finding B — "What Predicts Growth?" (ML Appendix, p.28).** Logistic Regression reports 71%
holdout accuracy separating growing from declining pages, evaluated with an 80/20 split (stated
plainly in Methodology, p.36: *"Random Forest (80/20 split), Logistic Regression (80/20 split)"*
— no mention of grouping by brand).

**My methodology question:** with 57 brands in the full study (and content clearly correlated
within a brand — shared templates, shared editorial calendar, shared traffic swings, same as my
own dataset's `client_id` grouping), was this 80/20 split row-level random or grouped by brand? If
it's row-level random, some of that 71% could be the model partly recognizing a brand it already
saw pages from in training, rather than truly generalizing to an unseen brand — the exact gap I
measure directly on my own model in Section 2. A second, smaller question: 71% accuracy needs the
class base rate next to it to mean anything (the skill's rule: "accuracy of 71% on a label that's
62% positive is 9 points of skill, not 71") — the paper doesn't state the growing/declining split
for this cut, so a reader can't tell how much of the 71% is base-rate.

Both questions are asked in the same spirit the paper asks of itself elsewhere — it already
discloses limitations like "Health score is a FlyRank composite metric, not a Google-endorsed
standard" and "Observational study: correlations do not prove causation." These two questions just
extend that same disclosed standard one level further, the way I'd want my own Week-5 notebook
read next.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/umairhussainn/ml-internship"
REPO_DIR = "ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

SEED = 42
pd.set_option("display.width", 120)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
for col in ["search_volume", "competition", "cpc", "word_count", "char_count"]:
    df[f"has_{col}"] = df[col].notna().astype(int)

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_search_volume", "has_competition", "has_cpc", "has_word_count", "has_char_count",
]
CATEGORICAL_FEATURES = ["competition_level", "content_type", "main_intent", "position_tier"]

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order][:k].mean())

def make_pipeline(numeric_feats, categorical_feats):
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_feats),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_feats),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])

print("Setup complete —", df.shape[0], "rows,", df["client_id"].nunique(), "clients")

Setup complete — 30000 rows, 32 clients


## 2. My model under an honest split (before/after)

Same question I just asked the paper, run on my own work: my Week-5 comparison already used a
client-grouped split. Here I build the dishonest counterfactual — a plain random row-level 80/20
split, the same kind of split the paper's Methodology section names for its own logistic
regression — using the same features, same model (Logistic Regression, my Week-5 winner), same
metric (`precision_at_50`), so the only thing that changes is the split.

In [ ]:
def run_split(train_idx, test_idx, label):
    train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]
    X_train, y_train = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES], train_df["is_declining_label"]
    X_test, y_test = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES], test_df["is_declining_label"]
    pipe = make_pipeline(NUMERIC_FEATURES, CATEGORICAL_FEATURES).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    p50 = precision_at_k(y_test.values, proba, 50)
    overlap = len(set(train_df["client_id"]) & set(test_df["client_id"]))
    return p50, overlap, len(train_df), len(test_df)

# BEFORE -- plain random row-level split (what the paper's Methodology names for its own model)
before_idx = next(ShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED).split(df))
p50_before, overlap_before, n_tr_b, n_te_b = run_split(*before_idx, "before")

# AFTER -- client-grouped split (same design as my Week-5 notebook, same seed)
after_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED).split(df, groups=df["client_id"]))
p50_after, overlap_after, n_tr_a, n_te_a = run_split(*after_idx, "after")

print(f"BEFORE -- random row split:    precision@50={p50_before:.3f}  client overlap train/test={overlap_before} clients")
print(f"AFTER  -- client-grouped split: precision@50={p50_after:.3f}  client overlap train/test={overlap_after} clients")
print(f"\nGap: {p50_before - p50_after:+.3f} precision@50 points came from letting the same client's other pages leak into training.")

BEFORE -- random row split:    precision@50=0.880  client overlap train/test=31 clients
AFTER  -- client-grouped split: precision@50=0.740  client overlap train/test=0 clients

Gap: +0.140 precision@50 points came from letting the same client's other pages leak into training.


**Reading the gap.** The random split scores 0.14 precision@50 points higher than the grouped
split (0.88 vs 0.74) with 31 of 32 clients appearing on both sides of the random split — almost
every client leaks. That gap is not model skill; it's the model partly recognizing a client it
already saw pages from during training. The grouped number (0.740, zero client overlap) is the one
I reported in Week 5 and the one I stand behind. This is precisely the question I raised about the
paper's own 71%-accuracy claim in Section 1, Finding B — now demonstrated on my own data instead of
just asked about someone else's.

## 3. Leakage audit

Repeating the Week-3-style hunt (label-derived features, overlapping windows, product/decision
flags) on my final Week-5 feature set, plus the skill's own verification step: deliberately inject
a known-leaky feature and confirm the score reacts the way leakage predicts it should.

In [ ]:
train_idx, test_idx = after_idx  # reuse the honest, client-grouped split from Section 2
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

def fit_eval(numeric_feats, categorical_feats):
    X_train, y_train = train_df[numeric_feats + categorical_feats], train_df["is_declining_label"]
    X_test, y_test = test_df[numeric_feats + categorical_feats], test_df["is_declining_label"]
    pipe = make_pipeline(numeric_feats, categorical_feats).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    return precision_at_k(y_test.values, proba, 50)

honest_p50 = fit_eval(NUMERIC_FEATURES, CATEGORICAL_FEATURES)

# Test 1 -- label-derived feature. trend_pct literally DEFINES trend_direction, which DEFINES
# is_declining_label. It's excluded from my real feature set (Section 1 of my Week-5 notebook
# confirms this) -- here I inject it on purpose to prove my test harness would actually catch it.
leaky_p50 = fit_eval(NUMERIC_FEATURES + ["trend_pct"], CATEGORICAL_FEATURES)

# Test 2 -- overlapping-window check. impressions_90d / clicks_90d are 90-day totals that CONTAIN
# the last_30d/prev_30d windows the label is built from. Do they inflate the score the way a real
# window-overlap leak would?
no_overlap_feats = [f for f in NUMERIC_FEATURES if f not in ("log_impressions_90d", "log_clicks_90d")]
no_overlap_p50 = fit_eval(no_overlap_feats, CATEGORICAL_FEATURES)

print(f"Honest feature set (Week-5, no label-derived columns): precision@50 = {honest_p50:.3f}")
print(f"+ trend_pct injected on purpose (label source):        precision@50 = {leaky_p50:.3f}   (jump: {leaky_p50-honest_p50:+.3f})")
print(f"- log_impressions_90d / log_clicks_90d removed:        precision@50 = {no_overlap_p50:.3f}   (change: {no_overlap_p50-honest_p50:+.3f})")

Honest feature set (Week-5, no label-derived columns): precision@50 = 0.740
+ trend_pct injected on purpose (label source):        precision@50 = 1.000   (jump: +0.260)
- log_impressions_90d / log_clicks_90d removed:        precision@50 = 0.720   (change: -0.020)


**Reading the two tests.**

- **Test 1 confirms the harness works.** Injecting `trend_pct` jumps precision@50 from 0.740 to
  1.000 — a perfect score, the exact "confession" pattern the skill describes. This is not a real
  result; it's proof that if this kind of leak were hiding in my real feature set, this test would
  catch it. It isn't hiding — `trend_pct` and `trend_direction` are confirmed absent from
  `NUMERIC_FEATURES`/`CATEGORICAL_FEATURES` (checked explicitly in my Week-5 notebook, Section 1).
- **Test 2 is a real, honest negative.** Removing `impressions_90d`/`clicks_90d` — the two features
  whose 90-day window mechanically contains the 30-day windows the label is built from — barely
  moves the score (0.740 → 0.720). If these were doing meaningful label-window leaking, I'd expect
  a collapse closer to what Test 1 showed. A small, non-collapsing change is consistent with these
  columns carrying real, mostly-legitimate signal about overall visibility, not smuggled label
  information.

**Attack checklist:**
- [x] Timeline drawn: this is a static snapshot (not a forward-looking prediction), so "before the
      label window" means "not derived from `trend_direction`/`trend_pct`" — confirmed absent.
- [x] No label-derived or sibling columns in the features — confirmed by Test 1's harness check.
- [x] No product flags / existing-system scores as features — my Week-4 `baseline_score` and the
      reference pipeline's `baseline_refresh_score` are used only as the comparison baseline in
      Week 5, never as a model input.
- [x] Split grouped by client (`client_id`) — Section 2.
- [x] Base rate printed next to every metric — Week-5 comparison table includes
      `base_rate_always_positive` (0.511) next to every model score.
- [x] Top feature importance sanity-checked — Week-5 Section 4 already checked permutation
      importance wasn't suspiciously dominated by one feature; Test 2 here extends that check.
- [x] Metrics recomputed out-of-fold — the grouped-split test set was never seen during training
      for any model reported here or in Week 5.

## 4. Claim rewrite

My boldest sentence from Week 5 (Section 4, "Practical read"):

> *"the fix isn't 'pick a fancier model' — it's adding a feature that distinguishes 'no clicks
> because of a featured snippet' from 'no clicks because the page is declining.'"*

That's stated as a settled conclusion. Rewritten in safe language, with the honest-split gap and
small test size folded in:

In [ ]:
claim_before = (
    "the fix isn't 'pick a fancier model' -- it's adding a feature that distinguishes "
    "'no clicks because of a featured snippet' from 'no clicks because the page is declining.'"
)

claim_after = (
    "Observed on this client-grouped holdout (7 clients, all 'keyword article' content -- see "
    "Week-5 Section 2 caveat): Logistic Regression's top-50 false positives were directionally "
    "concentrated among well-ranked, zero-CTR pages, a pattern that also appeared in my Week-4 "
    "rule-based baseline's own top picks. This is decision-support for one hypothesis -- a missing "
    "'likely featured snippet' flag -- not a validated fix; I have not tested whether adding such a "
    "feature would measurably change precision@50, and the honest-split-vs-random gap in Section 2 "
    "(0.740 vs 0.880) is a reminder that any single number here should be read as directional, not "
    "as a stable production estimate, until it's confirmed on a larger, more representative holdout."
)

print("BEFORE (overclaimed):\n", claim_before)
print("\nAFTER (safe language):\n", claim_after)

BEFORE (overclaimed):
 the fix isn't 'pick a fancier model' -- it's adding a feature that distinguishes 'no clicks because of a featured snippet' from 'no clicks because the page is declining.'

AFTER (safe language):
 Observed on this client-grouped holdout (7 clients, all 'keyword article' content -- see Week-5 Section 2 caveat): Logistic Regression's top-50 false positives were directionally concentrated among well-ranked, zero-CTR pages, a pattern that also appeared in my Week-4 rule-based baseline's own top picks. This is decision-support for one hypothesis -- a missing 'likely featured snippet' flag -- not a validated fix; I have not tested whether adding such a feature would measurably change precision@50, and the honest-split-vs-random gap in Section 2 (0.740 vs 0.880) is a reminder that any single number here should be read as directional, not as a stable production estimate, until it's confirmed on a larger, more representative holdout.
